In [ ]:
from time import sleep
from pynq.overlays.base import BaseOverlay
from pynq import GPIO
import pynq
import subprocess as sp
import netifaces as ni
import socket
import struct
import numpy as np
from doa_mts import *
import scipy.io
import os


# oled.write("RFSoC-PYNQ\nVersion {}".format(pynq.__version__))

# # LMK clock config
# lmk_reset = GPIO(341, 'out')
# lmk_clk_sel0 = GPIO(342, 'out')
# lmk_clk_sel1 = GPIO(346, 'out')

# lmk_reset.write(1)
# lmk_reset.write(0)
# lmk_clk_sel0.write(0)
# lmk_clk_sel1.write(0)

ps_btn = GPIO(357, 'in') #  GPIO.get_gpio_base() + pin num

o1 = doaMtsOverlay('doa_mts.bit')

mat = scipy.io.loadmat('/boot/data.mat')

data = mat.get('sigInt16')
o1.dac_data_mem_write(data[0], o1.dac0_player)
o1.dac_data_mem_write(data[0], o1.dac1_player)





In [ ]:
import asyncio, socket, struct
import numpy as np

async def handle_matlab(reader: asyncio.StreamReader, writer: asyncio.StreamWriter, o1, splitSym=";"):
    sock = writer.get_extra_info('socket')
    sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_KEEPALIVE, 1)

    count = int(o1.data_size)

    try:
        while True:
            # Try to read small control packet without blocking too long
            try:
                # Adjust size to your control protocol; 0 means poll-like
                ctrl = await asyncio.wait_for(reader.read(0), timeout=0.0)
            except asyncio.TimeoutError:
                pass
            except Exception:
                pass

            # If you have a defined control message size or delimiter,
            # implement it with reader.readexactly or reader.readline here,
            # then call o1.handle_commands(...)

            if o1.dataStream:
                data = o1.get_custom_data(count)
                if isinstance(data, np.ndarray) and data.dtype == np.int16 and data.flags['C_CONTIGUOUS']:
                    payload = memoryview(data)
                else:
                    payload = memoryview(np.asarray(data, dtype=np.int16))
                writer.write(payload)
                await writer.drain()

            # Optional tiny pause to yield the event loop
            await asyncio.sleep(0)
    except (asyncio.IncompleteReadError, ConnectionResetError, BrokenPipeError):
        pass
    finally:
        writer.close()
        await writer.wait_closed()

async def main(host, port, o1):
    server = await asyncio.start_server(
        lambda r, w: handle_matlab(r, w, o1),
        host, port, reuse_address=True
    )
    addrs = ", ".join(str(s.getsockname()) for s in server.sockets)
    print(f"Serving on {addrs}")
    async with server:
        await server.serve_forever()

# asyncio.run(main("0.0.0.0", 9000, o1))


In [ ]:
asyncio.run(main(os.environ["RFSOC_BIND_IP"], 4000, o1))


In [ ]:
asyncio.run(main(os.environ["RFSOC_BIND_IP"], 4000, o1))


In [ ]:
asyncio.run(main(os.environ["RFSOC_BIND_IP"], 4000, o1))


In [ ]:
asfasf

In [ ]:
try:
    iface = ni.gateways()['default'][ni.AF_INET][1]
    ip_address = ni.ifaddresses(iface)[2][0]['addr']    
except:
    if ('usb0' in ni.interfaces()) and (len(ni.ifaddresses('usb0')) == 3):
        iface = 'usb0'
        ip_address = ni.ifaddresses('usb0')[2][0]['addr']
    else:
        oled.write("RFSoC-PYNQ\nNo IP detected")
oled.write("IP Addr({}):\n{}".format(iface, ip_address))

server_ip = ip_address  # Use the appropriate IP address or hostname
server_port = 4000  # Use a free port number

server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# Bind the socket to the server IP and port
server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server_socket.bind((server_ip, server_port))
oled.write("TCP/IP server on\n Interface({})".format(iface))

server_socket.listen(1)
client_socket, client_address = server_socket.accept()
oled.write("Connected")

In [ ]:
while True:
    if ps_btn.read() == 1:
        oled.write("PS BTN reset\nServer Shutdown")
        break
    try:
        while True:
            commandList = o1.handle_client_connection(client_socket).split(splitSym)
            o1.handle_commands(commandList)
            # test = o1.dataStream
            # %store test
            # print(o1.dataStream)
            # while o1.dataStream:    
            if o1.dataStream:    
                data = o1.get_custom_data(o1.data_size)
                data_binary = struct.pack('h' * o1.data_size, *data)
                client_socket.sendall(data_binary)
    except:
        print('Reconnect or Error')
        server_socket.listen(1)
        client_socket, client_address = server_socket.accept()
        oled.write("Reconnect or Error")